# Maps of the Landsat 30 m monthly indices over the Ogallala aquifer: 4 km, three indices per figure

Reads the yearly files written by `Landsat_Indices_30m_Monthly_Ogallala.ipynb` from Drive, averages the 30 m pixels
of **one month** into 4 km cells for the whole aquifer, and draws every index (3 side by side per figure) with the
aquifer polygon on top. Figures are saved to Drive; the 4 km grid is saved too, so a re-plot does not touch the 30 m files.

| | |
|---|---|
| Input | `MyDrive/MSUGWB/LANDSAT_C2L2_indices_30m_monthly_tiles_v2/<year>/<tile>/*.nc` (and `frames.json`) |
| Output | `MyDrive/MSUGWB/LANDSAT_indices_4km_maps/<YYYY-MM>/` : `*_figN_<three indices>.png` and `*_4km_<YYYY-MM>.npz` |
| Choose | `YEAR` and `MONTH` in Cell 1 (`MONTH = None` picks the month that most tiles have finished) |
| Runtime | CPU is enough; no Earth Engine, no GPU |

**Run:** *Run all*. Cell 3 first prints, for the chosen year, how many tiles hold each month, so you can pick a month for
which every tile is done. If some tiles are not finished the map is still drawn, with those tiles hatched and the count in
the footer; set `REQUIRE_COMPLETE = True` to refuse instead.

**How it stays fast**
* Only the chosen month is read (the files are chunked one month x 512 x 512 pixels), only the indices you ask for, and only
  the chunk columns that contain aquifer pixels. Integers are read as stored (no float conversion) and summed in
  16 x 16 pixel blocks.
* Tiles are processed in parallel (`N_WORKERS` processes), because opening files on Drive is the slow part.
* The 4 km result is cached on Drive and reused when the same tiles hold the month.

**What the numbers are**
* A 4 km cell is the **mean of the monthly values of its valid 30 m pixels** (each of which is already a monthly median).
  Pixels without a usable observation are left out; a cell needs `MIN_VALID_PIXELS` of them to be drawn.
* Tiles are in their own UTM zone grids. Each 480 m block of pixels is moved to the 4 km Albers grid (`EPSG:5070`) by its
  centre, so a pixel can be up to 240 m from the cell it is counted in; nothing at 30 m is resampled.
* Along the zone 13/14 boundary both zones hold the same ground; a cell there pools the pixels of both.
* Grey inside the outline: no valid pixel. Hatched: the tile has not been processed for that month.
* Colours: one hue (light to dark) for magnitude indices; indices that change sign at 0 (water, moist canopy) use two hues
  around a pale midpoint. The range of each map is set by `color_limits()` in Cell 4 (default: 2nd-98th percentile of the
  map's own cells; override per index with `COLOR_LIMITS`).


In [ ]:
# ═══════════════ CELL 1: install, mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, sys, subprocess, zipfile
try:
    import netCDF4
except ImportError:                       # Colab does not ship netCDF4
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "netCDF4"], check=True)
    import netCDF4
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'      # same shapefile the processing notebook used
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
YEAR  = 2022          # year to map
MONTH = 7             # 1-12; or None -> the month of YEAR that most tiles have finished (ties: the earliest)
REQUIRE_COMPLETE = False   # True -> stop unless every tile of the aquifer has that month; False -> map what exists
                           # and mark the missing tiles with hatching

INDICES_TO_PLOT = None     # None -> every index found in the files; or e.g. ["NDVI", "NDMI", "MNDWI"]
PER_FIGURE      = 3        # maps per figure (side by side)

CELL_M   = 4000            # output cell size in metres: the 30 m pixels are averaged into cells this big
GRID_CRS = "EPSG:5070"     # equal-area projection the map is drawn in (CONUS Albers)
MIN_VALID_PIXELS = 50      # a cell needs at least this many valid 30 m pixels (of ~17,800) to be drawn.
                           # Mostly matters along the polygon edge, where a cell can hold a handful of pixels

USE_TEST_FOLDER = False    # True -> read the processing notebook's rehearsal output (.../_test)
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
PRODUCT_TAG = "LANDSAT_C2L2_indices_30m_monthly_tiles_v2"      # folder + file prefix of the processing notebook
PLOT_DIR    = f"{DRIVE_ROOT}/LANDSAT_indices_4km_maps"          # figures and the 4 km cache are saved here

# ---- speed ----
N_WORKERS = 4              # tiles read at once (processes). Drive is slow to open files, so more than the
                           # 2 CPUs of a free Colab runtime still helps. 1 -> no parallelism
SUB_PX    = 16             # 30 m pixels are first summed in blocks of SUB_PX x SUB_PX (480 m) and the blocks are then
                           # assigned to the 4 km cells; must divide the files' 512-pixel chunk edge
USE_CACHE = True           # reuse the 4 km result of an earlier run when the same tiles hold the month

# ---- look ----
DPI = 200                  # of the saved PNGs
PERCENTILES = (2, 98)      # default colour range: these percentiles of the 4 km cells (see color_limits() in Cell 4)
COLOR_LIMITS = {}          # fixed colour range per index, e.g. {"NDVI": (0, 0.9), "MNDWI": (-0.5, 0.5)}
# =======================================================================

In [ ]:
# ═══════════════ CELL 2: paths, aquifer polygon, 4 km grid, and the per-tile reader ═══════════════
import re, glob, json, math, time, shutil, calendar, textwrap, tempfile
import datetime as dt
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely

ROI_LABEL = "Ogallala"
DATA_DIR  = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if USE_TEST_FOLDER else "")
OUT_DIR   = f"{PLOT_DIR}" + ("_test" if USE_TEST_FOLDER else "")
TILE_RE   = re.compile(r"z\d\de\d+n\d+")
EPOCH     = dt.date(1970, 1, 1)
PIXEL     = 30.0
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"{DATA_DIR} does not exist"

# ---------- aquifer polygon and the common 4 km grid ----------
_roi = gpd.read_file(SHP_PATH)
if _roi.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")


def _union(gs):
    return gs.union_all() if hasattr(gs, "union_all") else gs.unary_union


ROI_WGS = _union(_roi.geometry)                                  # the polygon in the shapefile's own CRS
ROI_G   = _union(_roi.to_crs(GRID_CRS).geometry)                 # ... and in the map's CRS
_minx, _miny, _maxx, _maxy = ROI_G.bounds
GRID = dict(crs=GRID_CRS, cell=float(CELL_M),
            x0=math.floor(_minx / CELL_M - 1) * CELL_M, y0=math.ceil(_maxy / CELL_M + 1) * CELL_M)
GRID["nx"] = math.ceil((_maxx - GRID["x0"]) / CELL_M + 1)
GRID["ny"] = math.ceil((GRID["y0"] - _miny) / CELL_M + 1)
print(f"4 km grid: {GRID['nx']} x {GRID['ny']} cells in {GRID_CRS}; aquifer {ROI_G.area / 1e6:,.0f} km2")


# ---------- which files exist ----------
def tile_files(year):
    """{tile: path} of the yearly files on Drive: <data folder>/<year>/<tile>/<file>.nc"""
    out = {}
    for p in glob.glob(f"{DATA_DIR}/{year}/z*e*n*/{PRODUCT_TAG}_*_{year}_{ROI_LABEL}.nc"):
        key = os.path.basename(os.path.dirname(p))
        if TILE_RE.fullmatch(key):
            out[key] = p
    return out


def expected_frames():
    """{tile: frame} of every tile that holds aquifer pixels (frames.json, written by the processing notebook)."""
    p = f"{DATA_DIR}/frames.json"
    if not os.path.exists(p):
        return {}
    with open(p) as f:
        return {k: v for k, v in json.load(f).get("frames", {}).items() if v}


def month_days(year, month):
    return (dt.date(year, month, 1) - EPOCH).days


# ---------- parallel helper ----------
def run_parallel(fn, jobs, workers, label):
    """fn(job) for every job; processes (fork) when possible, else one after the other. Order is kept.
    netCDF4/HDF5 is not thread-safe, so threads are not an option."""
    jobs, out, t0 = list(jobs), {}, time.time()
    if workers > 1 and len(jobs) > 1 and "fork" in mp.get_all_start_methods():
        with ProcessPoolExecutor(min(workers, len(jobs)), mp_context=mp.get_context("fork")) as ex:
            futs = {ex.submit(fn, j): i for i, j in enumerate(jobs)}
            for n, f in enumerate(as_completed(futs), 1):
                out[futs[f]] = f.result()
                print(f"\r{label}: {n}/{len(jobs)} ({time.time() - t0:.0f} s)", end="", flush=True)
    else:
        for i, j in enumerate(jobs):
            out[i] = fn(j)
            print(f"\r{label}: {i + 1}/{len(jobs)} ({time.time() - t0:.0f} s)", end="", flush=True)
    print()
    return [out[i] for i in range(len(jobs))]


def _retry(fn, *args, tries=3):
    """Drive's mount sometimes fails to open a file once; try again before giving up (returns the error as text)."""
    err = None
    for k in range(tries):
        try:
            return fn(*args)
        except (OSError, RuntimeError) as e:
            err = repr(e)
            time.sleep(5 * (k + 1))
    return dict(error=err)


# ---------- cheap pass: which months does a file hold ----------
def _months_in(path, year):
    ds = nc4.Dataset(path)
    try:
        t = np.asarray(ds.variables["time"][:], "f8")
    finally:
        ds.close()
    days = [EPOCH + dt.timedelta(days=int(d)) for d in t]
    return dict(months=sorted(d.month for d in days if d.year == year))


def scan_months(job):
    return _retry(_months_in, *job)


# ---------- the main pass: one tile, one month, all indices -> sums on 480 m blocks, placed on the 4 km grid ----------
def _coarsen_tile(path, target_days, grid, sub, wanted):
    ds = nc4.Dataset(path)
    try:
        ds.set_auto_maskandscale(False)            # raw int16: no float masked arrays, much faster
        t = np.asarray(ds.variables["time"][:], "f8")
        hit = np.flatnonzero(t == target_days)
        if not hit.size:
            return dict(missing=True)
        ti = int(hit[0])
        x = np.asarray(ds.variables["x"][:], "f8")
        y = np.asarray(ds.variables["y"][:], "f8")
        h, w = y.size, x.size
        found = [n for n, v in ds.variables.items() if hasattr(v, "formula") and v.dimensions == ("time", "y", "x")]
        names = found if wanted is None else [n for n in wanted if n in found]
        if not names:
            return dict(error=f"none of the wanted indices is in the file (it holds {found})")
        vs = [ds.variables[n] for n in names]
        fills = [int(v.getncattr("_FillValue")) for v in vs]
        scales = [float(v.getncattr("scale_factor")) for v in vs]
        cy, cx = vs[0].chunking()[1:]
        strip = max(sub, cy // sub * sub)                             # rows per read: whole chunks, whole blocks
        mask = np.asarray(ds.variables["roi_mask"][:]).astype(bool) if "roi_mask" in ds.variables else None

        nby, nbx = -(-h // sub), -(-w // sub)
        S = np.zeros((nby, nbx, len(names)))                          # sum of the scaled values per block
        C = np.zeros((nby, nbx, len(names)), np.int32)                # valid 30 m pixels per block
        for r0 in range(0, h, strip):
            r1 = min(h, r0 + strip)
            c0, c1 = 0, w
            if mask is not None:                                      # read only the chunk columns that hold aquifer
                cols = np.flatnonzero(mask[r0:r1].any(axis=0))
                if not cols.size:
                    continue
                c0 = cols[0] // cx * cx
                c0 -= c0 % sub
                c1 = min(w, -(-(int(cols[-1]) + 1) // cx) * cx)
            ph, pw = -(r1 - r0) % sub, -(c1 - c0) % sub
            for k, v in enumerate(vs):
                a = v[ti, r0:r1, c0:c1]
                ok = a != fills[k]
                a = np.where(ok, a, np.int16(0))
                if ph or pw:
                    a, ok = np.pad(a, ((0, ph), (0, pw))), np.pad(ok, ((0, ph), (0, pw)))
                n_r, n_c = a.shape[0] // sub, a.shape[1] // sub
                s = a.reshape(n_r, sub, n_c, sub).sum(axis=(1, 3), dtype=np.int32)
                c = ok.reshape(n_r, sub, n_c, sub).sum(axis=(1, 3), dtype=np.int32)
                br, bc = r0 // sub, c0 // sub
                S[br:br + n_r, bc:bc + n_c, k] = s * scales[k]
                C[br:br + n_r, bc:bc + n_c, k] = c

        # centre of every block (mean of the pixel centres it really holds) -> grid cell
        def centres(c, n):
            pad = np.full(n * sub, np.nan)
            pad[:c.size] = c
            return np.nanmean(pad.reshape(n, sub), axis=1)
        bx, by = np.meshgrid(centres(x, nbx), centres(y, nby))
        crs_v = ds.variables["crs"]
        crs = pyproj.CRS.from_wkt(crs_v.spatial_ref) if hasattr(crs_v, "spatial_ref") else json.loads(ds.frame)["crs"]
        gx, gy = pyproj.Transformer.from_crs(crs, grid["crs"], always_xy=True).transform(bx.ravel(), by.ravel())
        ix = np.floor((gx - grid["x0"]) / grid["cell"]).astype(np.int64)
        iy = np.floor((grid["y0"] - gy) / grid["cell"]).astype(np.int64)
        inside = (ix >= 0) & (ix < grid["nx"]) & (iy >= 0) & (iy < grid["ny"])
        S, C = S.reshape(-1, len(names)), C.reshape(-1, len(names))
        keep = inside & (C.sum(axis=1) > 0)                           # only blocks with data travel back to the parent
        return dict(names=names, available=found, cell=(iy * grid["nx"] + ix)[keep].astype(np.int64), S=S[keep], C=C[keep],
                    long_names=[getattr(v, "long_name", n) for n, v in zip(names, vs)],
                    harmonisation=getattr(ds, "sensor_harmonisation", ""))
    finally:
        ds.close()


def coarsen_tile(job):
    return _retry(_coarsen_tile, *job)

In [ ]:
# ═══════════════ CELL 3: what has been processed, choose the month, build the 4 km grid ═══════════════
t_start = time.time()
files = tile_files(YEAR)
if not files:
    have_years = sorted(os.path.basename(d) for d in glob.glob(f"{DATA_DIR}/[0-9][0-9][0-9][0-9]"))
    raise FileNotFoundError(f"no yearly files for {YEAR} under {DATA_DIR} (years there: {have_years})")
frames = expected_frames()
expected = sorted(set(frames) | set(files))
print(f"{len(files)} tile files for {YEAR}; {len(expected)} tiles expected"
      + ("" if frames else "  (frames.json not found: the expected set is just the files that exist)"))

# --- which months does each tile hold (a header read per file, in parallel) ---
scan = dict(zip(sorted(files), run_parallel(scan_months, [(files[k], YEAR) for k in sorted(files)], N_WORKERS, "reading time axes")))
bad = {k: v["error"] for k, v in scan.items() if "error" in v}
if bad:
    print("could not open:", bad)
held = {k: set(v["months"]) for k, v in scan.items() if "months" in v}

print(f"\n{YEAR}: tiles that hold the month (of {len(expected)})")
cover = {m: sorted(k for k, ms in held.items() if m in ms) for m in range(1, 13)}
for m in range(1, 13):
    n = len(cover[m])
    print(f"  {calendar.month_abbr[m]}  {n:3d}  {'#' * round(30 * n / len(expected)):<30s} " + ("complete" if n == len(expected) else ""))

if MONTH is None:
    MONTH = max(range(1, 13), key=lambda m: (len(cover[m]), -m))
    print(f"\nMONTH = None -> {calendar.month_name[MONTH]} ({len(cover[MONTH])} tiles)")
tiles_done = cover[MONTH]
tiles_missing = [k for k in expected if k not in tiles_done]
print(f"\nMapping {calendar.month_name[MONTH]} {YEAR}: {len(tiles_done)} of {len(expected)} tiles"
      + (f"; missing {len(tiles_missing)}: {', '.join(tiles_missing[:12])}{' ...' if len(tiles_missing) > 12 else ''}" if tiles_missing else ""))
if not tiles_done:
    raise RuntimeError("no tile holds that month; pick one from the table above")
if tiles_missing and REQUIRE_COMPLETE:
    raise RuntimeError("REQUIRE_COMPLETE is on and tiles are missing (see above)")

# --- coarsen: all 30 m pixels of the month -> 4 km cells (or reuse the cache) ---
tag = f"{YEAR}-{MONTH:02d}"
os.makedirs(f"{OUT_DIR}/{tag}", exist_ok=True)
CACHE = f"{OUT_DIR}/{tag}/{ROI_LABEL}_Landsat30m_indices_{CELL_M // 1000}km_{tag}.npz"
sig = json.dumps(dict(tiles=tiles_done, grid=GRID, sub=SUB_PX, product=PRODUCT_TAG), sort_keys=True)

M = Cn = names = long_names = harmonisation = None
if USE_CACHE and os.path.exists(CACHE):
    z = np.load(CACHE, allow_pickle=False)
    c_names = [str(n) for n in z["names"]]
    if str(z["signature"]) == sig and (INDICES_TO_PLOT is None or set(INDICES_TO_PLOT) <= set(c_names)):
        names = c_names if INDICES_TO_PLOT is None else [n for n in INDICES_TO_PLOT]
        sel = [c_names.index(n) for n in names]
        M, Cn = z["mean"][sel], z["count"][sel]
        long_names = [str(z["long_names"][i]) for i in sel]
        harmonisation = str(z["harmonisation"])
        print(f"Reused the 4 km result from {CACHE}")
    else:
        print("The cache on Drive is for other tiles/settings; it is rebuilt.")

if M is None:
    wanted = None if INDICES_TO_PLOT is None else list(INDICES_TO_PLOT)
    res = run_parallel(coarsen_tile, [(files[k], month_days(YEAR, MONTH), GRID, SUB_PX, wanted) for k in tiles_done],
                       N_WORKERS, "coarsening tiles")
    errs = {k: r["error"] for k, r in zip(tiles_done, res) if "error" in r}
    if errs:
        raise RuntimeError(f"tiles that could not be read (nothing is plotted from a partial read): {errs}")
    res = [r for r in res if not r.get("missing")]
    names, long_names, harmonisation = res[0]["names"], res[0]["long_names"], res[0]["harmonisation"]
    if any(r["names"] != names for r in res):
        raise RuntimeError("the tiles hold different sets of indices; the files come from different runs")
    if wanted and set(wanted) - set(res[0]["available"]):
        raise KeyError(f"not in the files: {sorted(set(wanted) - set(res[0]['available']))}; they hold {res[0]['available']}")
    ncell, nv = GRID["nx"] * GRID["ny"], len(names)
    Ssum, Csum = np.zeros((nv, ncell)), np.zeros((nv, ncell), np.int64)
    for r in res:                                                     # pixels of every tile land in their 4 km cell
        for k in range(nv):
            Ssum[k] += np.bincount(r["cell"], weights=r["S"][:, k], minlength=ncell)
            Csum[k] += np.bincount(r["cell"], weights=r["C"][:, k], minlength=ncell).astype(np.int64)
    with np.errstate(invalid="ignore", divide="ignore"):
        M = np.where(Csum > 0, Ssum / Csum, np.nan).astype("f4").reshape(nv, GRID["ny"], GRID["nx"])
    Cn = Csum.astype("i4").reshape(nv, GRID["ny"], GRID["nx"])
    tmp = os.path.join(tempfile.gettempdir(), "_indices_cache.npz")
    np.savez_compressed(tmp, mean=M, count=Cn, names=np.array(names), long_names=np.array(long_names),
                        harmonisation=harmonisation, signature=sig)
    shutil.copy(tmp, CACHE)
    os.remove(tmp)
    print(f"Saved the 4 km result: {CACHE} ({os.path.getsize(CACHE) / 1e6:.1f} MB)")

M = np.where(Cn >= MIN_VALID_PIXELS, M, np.nan)                       # the cell needs enough valid pixels
print(f"{len(names)} indices x {GRID['ny']} x {GRID['nx']} cells | {time.time() - t_start:.0f} s")

In [ ]:
# ═══════════════ CELL 4: maps, three indices per figure, saved to Drive ═══════════════
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path
from shapely.geometry.polygon import orient

INK, MUTED, NODATA = "#1f2328", "#59636e", "#e3e6ea"
# Magnitude indices get one hue, light -> dark; indices that change sign at 0 (water above 0, or moist canopy)
# get two hues around a pale neutral midpoint.
DIVERGING = {"NDMI", "NDWI", "MNDWI", "AWEInsh", "AWEIsh", "WI2015"}
SEQ_CMAP  = {"NMDI": "Blues", "GVMI": "Blues", "MSI": "Oranges"}      # every other index: Greens
DIV_CMAP  = "BrBG"                                                    # brown (dry / land) -> teal (moist / water)


def color_limits(name, values):
    """(vmin, vmax) of one index's colour scale, from its 4 km cells `values` (NaN = no data)."""
    if name in COLOR_LIMITS:
        return COLOR_LIMITS[name]
    # TODO(human): decide the colour-range policy. The default below stretches each map to its own 2nd-98th
    # percentile, which shows the spatial pattern well but means the same colour is a different value in every
    # month and index. Fixed physical ranges (via COLOR_LIMITS) make months comparable; this function is where
    # a middle way (e.g. fixed ranges for the vegetation indices, percentiles for the rest) would go.
    lo, hi = np.nanpercentile(values, PERCENTILES)
    if name in DIVERGING:
        m = max(abs(lo), abs(hi))
        return -m, m
    return lo, hi


# ---------- map furniture: polygon outline, grey under the polygon (= no usable data), unprocessed tiles ----------
extent = (GRID["x0"], GRID["x0"] + GRID["nx"] * GRID["cell"], GRID["y0"] - GRID["ny"] * GRID["cell"], GRID["y0"])
poly_gs = gpd.GeoSeries([ROI_G], crs=GRID_CRS)

hatch_geoms, _poly_in = [], {}
for key in tiles_missing:                                    # footprint of each unprocessed tile, clipped to the aquifer
    F = frames.get(key)
    if not F:
        continue
    if F["crs"] not in _poly_in:
        _poly_in[F["crs"]] = gpd.GeoSeries([ROI_WGS], crs=_roi.crs).to_crs(F["crs"]).iloc[0]
    box = shapely.geometry.box(F["x0"], F["y0"] - F["h"] * PIXEL, F["x0"] + F["w"] * PIXEL, F["y0"])
    part = shapely.segmentize(box.intersection(_poly_in[F["crs"]]), 1000.0)
    if not part.is_empty:
        hatch_geoms.append(gpd.GeoSeries([part], crs=F["crs"]).to_crs(GRID_CRS).iloc[0])
hatch_gs = gpd.GeoSeries(hatch_geoms, crs=GRID_CRS) if hatch_geoms else None


def clip_patch(geom, ax):
    """The aquifer polygon as a matplotlib clip path, so cells that straddle the boundary do not colour the outside."""
    verts, codes = [], []
    for part in (geom.geoms if hasattr(geom, "geoms") else [geom]):
        part = orient(part, 1.0)
        for ring in [part.exterior, *part.interiors]:
            xy = np.asarray(ring.coords)
            verts.extend(xy)
            codes += [Path.MOVETO] + [Path.LINETO] * (len(xy) - 2) + [Path.CLOSEPOLY]
    return PathPatch(Path(verts, codes), transform=ax.transData)


month_label = f"{calendar.month_name[MONTH]} {YEAR}"
cov = f"{len(tiles_done)} of {len(expected)} tiles processed"
saved = []
for f0 in range(0, len(names), PER_FIGURE):
    idx = list(range(f0, min(f0 + PER_FIGURE, len(names))))
    fig, axes = plt.subplots(1, len(idx), figsize=(5.4 * len(idx), 8.4), layout="constrained")
    fig.get_layout_engine().set(rect=(0, 0.05, 1, 0.95))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, idx):
        name, vals = names[k], M[k]
        poly_gs.plot(ax=ax, facecolor=NODATA, edgecolor="none", zorder=1)
        if np.isfinite(vals).any():
            vmin, vmax = color_limits(name, vals)
            if not vmax > vmin:
                vmin, vmax = vmin - 0.5, vmax + 0.5
            cmap = DIV_CMAP if name in DIVERGING else SEQ_CMAP.get(name, "Greens")
            im = ax.imshow(vals, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax,
                           interpolation="nearest", zorder=2)
            im.set_clip_path(clip_patch(ROI_G, ax))
            cb = fig.colorbar(im, ax=ax, orientation="horizontal", shrink=0.85, pad=0.01, aspect=32, extend="both")
            cb.outline.set_linewidth(0)
            cb.ax.tick_params(labelsize=9, colors=MUTED, length=2)
            cb.set_label(textwrap.fill(long_names[k], 44), fontsize=8.5, color=MUTED)
        else:
            ax.text(0.5, 0.5, "no valid cells", transform=ax.transAxes, ha="center", color=MUTED)
        if hatch_gs is not None:
            hatch_gs.plot(ax=ax, facecolor="none", edgecolor="#8b949e", hatch="////", linewidth=0, zorder=3)
        poly_gs.plot(ax=ax, facecolor="none", edgecolor=INK, linewidth=0.9, zorder=4)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_title(name, loc="left", fontsize=15, fontweight="bold", color=INK)
    fig.suptitle(f"Landsat monthly-median indices, {month_label}", x=0.01, ha="left", fontsize=17,
                 fontweight="bold", color=INK)
    handles = [Patch(facecolor=NODATA, edgecolor="none", label="inside the aquifer, no valid pixels"),
               Patch(facecolor="none", edgecolor=INK, label="aquifer boundary")]
    if hatch_gs is not None:
        handles.insert(1, Patch(facecolor="none", edgecolor="#8b949e", hatch="////", label="tile not processed yet"))
    fig.legend(handles=handles, loc="lower left", ncol=len(handles), frameon=False, fontsize=9,
               labelcolor=MUTED, bbox_to_anchor=(0.005, 0.0))
    fig.text(0.995, 0.012, f"Mean of 30 m pixels in {CELL_M // 1000} x {CELL_M // 1000} km cells ({GRID_CRS}) · {cov} · "
             "USGS Landsat C2 L2", ha="right", va="bottom", fontsize=8.5, color=MUTED)
    fname = f"{ROI_LABEL}_Landsat30m_indices_{CELL_M // 1000}km_{tag}_fig{f0 // PER_FIGURE + 1}_" + "-".join(names[i] for i in idx) + ".png"
    tmp = os.path.join(tempfile.gettempdir(), fname)
    fig.savefig(tmp, dpi=DPI, facecolor="white")                       # written locally, then copied: no half-written file on Drive
    shutil.copy(tmp, f"{OUT_DIR}/{tag}/{fname}")
    os.remove(tmp)
    saved.append(f"{OUT_DIR}/{tag}/{fname}")
    plt.show()
    plt.close(fig)

print(f"\n{len(saved)} figures saved in {OUT_DIR}/{tag}/")
for p in saved:
    print(" ", os.path.basename(p), f"({os.path.getsize(p) / 1e6:.1f} MB)")
if harmonisation:
    print("\nSensor harmonisation recorded in the files:", harmonisation[:160] + ("..." if len(harmonisation) > 160 else ""))